# Support Ticket Data Audit

Audit the registry-built canonical dataset, the downstream SFT and eval manifests, and the composition artifacts that describe provenance, label balance, null handling, and synthetic share.

In [ ]:
from collections import Counter
from pathlib import Path
from statistics import mean
import json
import sys

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

from json_ft.dataset_adapters import adapt_source_record, eval_manifest_record, messages_record, prompt_completion_record
from json_ft.utils import load_yaml, read_json, read_jsonl

canonical_path = repo_root / 'data' / 'manifests' / 'support_tickets_canonical.jsonl'
summary_path = repo_root / 'data' / 'manifests' / 'support_tickets_dataset_build_summary.json'
composition_path = repo_root / 'artifacts' / 'metrics' / 'support_tickets_dataset_composition.json'
model_config = load_yaml(repo_root / 'configs' / 'model.yaml')

rows = read_jsonl(canonical_path)
samples = [adapt_source_record(row, 'json_extraction') for row in rows]
summary = read_json(summary_path)
composition = read_json(composition_path)
base_model = model_config['base_model']['tokenizer_name_or_path']

print(f'Canonical rows: {len(samples)}')
print(f'Build profile: {summary["profile"]}')
print(f'Base tokenizer: {base_model}')

In [ ]:
split_counts = Counter(sample.split.value for sample in samples)
source_counts = Counter(sample.source_dataset for sample in samples)
issue_counts = Counter(sample.target.issue_category.value for sample in samples)
priority_counts = Counter(sample.target.priority.value for sample in samples)
product_counts = Counter(sample.target.product_area.value for sample in samples)
sentiment_counts = Counter(sample.target.sentiment.value for sample in samples)

print('Split counts:', dict(sorted(split_counts.items())))
print('Source counts:', dict(sorted(source_counts.items())))
print('Issue category counts:', dict(sorted(issue_counts.items())))
print('Priority counts:', dict(sorted(priority_counts.items())))
print('Product area counts:', dict(sorted(product_counts.items())))
print('Sentiment counts:', dict(sorted(sentiment_counts.items())))

In [ ]:
total = len(samples)
null_stats = {
    'customer.name': sum(1 for sample in samples if sample.target.customer.name is None) / total,
    'customer.account_id': sum(1 for sample in samples if sample.target.customer.account_id is None) / total,
    'customer.plan_tier': sum(1 for sample in samples if sample.target.customer.plan_tier is None) / total,
}
synthetic_count = sum(1 for sample in samples if sample.metadata.get('synthetic', False))

print('Percent null per nullable field:')
for field, value in sorted(null_stats.items()):
    print(f'  {field}: {value:.1%}')
print(f'Synthetic rows: {synthetic_count}/{total} ({synthetic_count/total:.1%})')
print('Source dominance share:', summary['source_dominance_share'])

In [ ]:
summary_lengths = [len(sample.target.summary) for sample in samples]
prompt_lengths = [len(prompt_completion_record(sample)['prompt']) for sample in samples if sample.split.value == 'train']
message_lengths = [sum(len(message['content']) for message in messages_record(sample)['messages']) for sample in samples if sample.split.value == 'train']

print('Summary length chars:', {'min': min(summary_lengths), 'max': max(summary_lengths), 'avg': round(mean(summary_lengths), 2)})
print('Prompt length chars:', {'min': min(prompt_lengths), 'max': max(prompt_lengths), 'avg': round(mean(prompt_lengths), 2)})
print('Messages length chars:', {'min': min(message_lengths), 'max': max(message_lengths), 'avg': round(mean(message_lengths), 2)})

try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    token_lengths = [len(tokenizer(prompt_completion_record(sample)['prompt'])['input_ids']) for sample in samples if sample.split.value == 'train']
    print('Prompt token lengths:', {'min': min(token_lengths), 'max': max(token_lengths), 'avg': round(mean(token_lengths), 2)})
except Exception as exc:
    token_lengths = []
    print('Skipping tokenizer-based prompt length audit:', exc)

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(summary_lengths, bins=min(6, len(summary_lengths)))
    axes[0].set_title('Summary Length Distribution')
    axes[0].set_xlabel('Characters')
    axes[1].hist(prompt_lengths, bins=min(6, len(prompt_lengths)))
    axes[1].set_title('Prompt Length Distribution')
    axes[1].set_xlabel('Characters')
    plt.tight_layout()
except Exception as exc:
    print('Skipping matplotlib plots:', exc)

In [ ]:
print('Composition rows:')
for row in composition['rows']:
    print(row)

print('\nAdapter reject examples:')
for row in summary.get('adapter_reject_examples', []):
    print(row)

In [ ]:
null_examples = [sample for sample in samples if sample.target.customer.name is None and sample.target.customer.account_id is None][:3]
print('Conservative null-assignment examples:')
for sample in null_examples:
    print('-' * 80)
    print(sample.record_id, sample.source_dataset)
    print(sample.input_text)
    print(sample.target.model_dump())

spot_check_examples = [sample for sample in samples if sample.source_dataset in {'cfpb_consumer_complaints', 'console_ai_it_helpdesk_synthetic_tickets'}][:3]
print('\nMapped rows to spot-check:')
for sample in spot_check_examples:
    print('-' * 80)
    print(sample.record_id, sample.source_dataset)
    print(sample.target.model_dump())

In [ ]:
sample = samples[0]
print('Canonical row:')
print(sample.model_dump_json(indent=2))

print('\nPrompt-completion export:')
print(json.dumps(prompt_completion_record(sample), indent=2, sort_keys=True))

print('\nMessages export:')
print(json.dumps(messages_record(sample), indent=2, sort_keys=True))

print('\nEval export:')
print(json.dumps(eval_manifest_record(sample), indent=2, sort_keys=True))